# 3. Organoid–Cell Relationship

## Purpose
This notebook assigns each single cell and nucleocentric object to its parent organoid,
then computes spatial relationship features (Euclidean distance, Mahalanobis distance,
and shell classification) for each cell relative to its parent organoid centroid.

This is **step 3 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
Parquet files from `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:
- `sc_profiles_{well_fov}.parquet` — merged Nuclei + Cell + Cytoplasm features (required)
- `organoid_profiles_{well_fov}.parquet` — organoid features (required)
- `nucleocentric_profiles_{well_fov}.parquet` — nucleocentric features (optional --
  treated as empty if this file doesn't exist, since not every pipeline that feeds
  this script produces deep-learning Nucleocentric data)

## Outputs
Three enriched parquet files written to `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`:

| File | Added columns |
|---|---|
| `sc_profiles_{well_fov}_related.parquet` | `ParentOrganoid`, shell/distance features |
| `organoid_profiles_{well_fov}_related.parquet` | `OrganoidSingleCellCount` |
| `nucleocentric_profiles_{well_fov}_related.parquet` | `ParentOrganoid` |

## Notes
- Parent organoid assignment uses bbox containment: a cell is assigned to the first
  organoid whose bounding box contains the cell's nuclear centroid.
- Spatial features are computed using Mahalanobis distance with a regularized covariance
  matrix (applied automatically when cell count is low).
- Shell classification divides cells into 4 concentric shells from organoid centroid
  outward, requiring at least 3 cells per shell.
- Object identifiers: accepts either the older CellProfiler-era pipeline's own
  `object_id` column (produced by IBP steps 00/0a/1/2) or ZEDProfiler's native
  `Metadata_Object_ObjectID`, normalized to `object_id` right after loading (see
  the data-loading cell below) rather than requiring a caller to rename it first.
  `image_set` is set directly from this script's own `well_fov` argument rather
  than required as an input column, since every row in a given input file
  belongs to the single well-FOV this script is invoked for.
- `nucleocentric_profiles_{well_fov}.parquet` is optional: if it doesn't exist,
  an empty dataframe is used instead of requiring a caller to manufacture a
  placeholder file. The output nucleocentric_profiles_{well_fov}_related.parquet
  is still always written (empty, if the input was empty) for schema consistency
  with the other two output files.

In [1]:
import os
import pathlib

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C4-1"
    image_based_profiles_subparent_name = "image_based_profiles"

### Pathing

In [3]:
# input paths
sc_profile_handcrafted_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_handcrafted_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)

sc_profile_sammed_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_sammed_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_sammed_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_sammed_profiles_{well_fov}.parquet"
).resolve(strict=True)
# Not strict=True like the other two inputs: ZEDProfiler produces no
# Nucleocentric (deep-learning) features at all, so a caller without any
# real Nucleocentric data has nothing to put here -- see the loading cell
# below, which falls back to an empty dataframe when this file is absent
# rather than requiring every caller to manufacture a placeholder file.
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
# output paths
sc_profile_handcrafted_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_handcrafted_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_sammed_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_sammed_related.parquet"
).resolve()
organoid_profile_sammed_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_sammed_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_handcrafted_output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
sc_profile_df = pd.read_parquet(sc_profile_handcrafted_path)
organoid_profile_df = pd.read_parquet(organoid_profile_handcrafted_path)
sc_profile_sammed_df = pd.read_parquet(sc_profile_sammed_path)
organoid_profile_sammed_df = pd.read_parquet(organoid_profile_sammed_path)
# ZEDProfiler-fed callers have no real Nucleocentric data to provide --
# fall back to an empty frame rather than requiring one. object_id is
# set here so the merge further down (which joins on object_id +
# image_set) has something to join against; image_set is set
# unconditionally for both dataframes just below regardless.
if nucleocentric_profile_path.exists():
    nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
else:
    nucleocentric_df = pd.DataFrame(columns=["object_id"])

# Normalize the object identifier column name: ZEDProfiler's own
# Metadata_Object_ObjectID is accepted directly, same object concept as
# the older CellProfiler-era pipeline's own `object_id` -- renamed once,
# here, rather than requiring a caller to disguise ZEDProfiler's column
# as the older convention before handoff. A no-op wherever `object_id`
# is already present.
for _df in (sc_profile_df, organoid_profile_df, nucleocentric_df):
    if "object_id" not in _df.columns and "Metadata_Object_ObjectID" in _df.columns:
        _df.rename(columns={"Metadata_Object_ObjectID": "object_id"}, inplace=True)

# `image_set` is just this well-FOV's own label -- this script already
# has it as `well_fov`, so set it directly rather than requiring it as an
# input column. Both dataframes are scoped to this single well-FOV
# already, so every row gets the same value.
sc_profile_df["image_set"] = well_fov
nucleocentric_df["image_set"] = well_fov

print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")
print(f"Single-cell sammed profile shape: {sc_profile_sammed_df.shape}")
print(f"Organoid sammed profile shape: {organoid_profile_sammed_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")

Single-cell profile shape: (55, 2667)
Organoid profile shape: (1, 889)
Single-cell sammed profile shape: (55, 9218)
Organoid sammed profile shape: (1, 3074)
Nucleocentric profile shape: (55, 3074)


In [5]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    # "area": CellProfiler-era naming (*_AreaSizeShape_*). "volumesizeshape":
    # ZEDProfiler's own naming (*_VolumeSizeShape_*) for the same measurement
    # family -- accepted directly so ZEDProfiler-sourced data needs no
    # column renaming to work with this notebook.
    if ("area" in x.lower() or "volumesizeshape" in x.lower())
    and "center" in x.lower()
    and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_AreaSizeShape_CenterX',
 'Nuclei_NoChannel_AreaSizeShape_CenterY',
 'Nuclei_NoChannel_AreaSizeShape_CenterZ']

In [6]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    # "area" (CellProfiler) or "volumesizeshape" (ZEDProfiler) -- see the
    # x_y_z_sc_colnames cell above for why both are accepted.
    if ("area" in x.lower() or "volumesizeshape" in x.lower())
    and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

# When sorted alphabetically, the bbox column names fall in this order:
#   [0] = *MaxX, [1] = *MaxY, [2] = *MaxZ, [3] = *MinX, [4] = *MinY, [5] = *MinZ
# This ordering is assumed in the bbox tuple construction below. Holds
# regardless of whether the matched family is AreaSizeShape or
# VolumeSizeShape: the family name prefix is identical across all six
# candidates, so sort order is determined only by the trailing Max/Min +
# axis letter.

In [7]:
# Initialize ParentOrganoid to -1 (sentinel for unassigned cells).
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array

# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Build the bbox tuple using the sorted column order documented in the cell above:
    # sorted alphabetically gives [MaxX, MaxY, MaxZ, MinX, MinY, MinZ]
    # so indices [5]=MinZ, [4]=MinY, [3]=MinX, [2]=MaxZ, [1]=MaxY, [0]=MaxX.
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # First-match-wins: if organoid bboxes overlap, a cell is assigned to the first
    # organoid whose bbox contains it and is never reassigned to a later one.
    # Both masks are NumPy arrays (positional) to avoid pandas index misalignment.
    unassigned_mask = sc_profile_df["ParentOrganoid"].values == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[sc_profile_df.index[final_mask], "ParentOrganoid"] = organoid_row[
        "object_id"
    ]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 1/1 [00:00<00:00, 1449.31it/s]

Assigned 55 cells to organoids
Unassigned cells: 0


### Add single-cell counts for each organoid

In [8]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="OrganoidSingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("OrganoidSingleCellCount")
organoid_profile_df.insert(2, "OrganoidSingleCellCount", sc_count)

### Carry the `ParentOrganoid` assignment and spatial features over to the sammed profiles, so that the same cells are assigned to the same organoids in both the

In [9]:
organoid_profile_sammed_df = organoid_profile_sammed_df.merge(
    organoid_profile_df[["object_id", "OrganoidSingleCellCount"]],
    left_on="object_id",
    right_on="object_id",
    how="left",
)

In [10]:
sc_profile_sammed_df = sc_profile_sammed_df.merge(
    sc_profile_df[["object_id", "ParentOrganoid"]],
    left_on="object_id",
    right_on="object_id",
    how="left",
)

### Empty dataframe fallbacks

If either the organoid or SC profile is empty for this well-FOV, a placeholder row
is inserted so that downstream merges always find consistent columns.

In [11]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["OrganoidSingleCellCount"] = (
    organoid_profile_df["OrganoidSingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C4-1,55,27421038.0,711.460877,936.634612,19.864556,50391450.0,267,1082,...,35.276193,36.687325,35.246483,36.669919,35.24153,35.241639,35.290421,36.683411,35.280867,35.286644


In [12]:
if organoid_profile_df.empty:
    # Write the empty DataFrame as-is. Parquet preserves schema (columns) even with
    # zero rows, so downstream union_by_name in 5.combining_profiles handles this
    # correctly. A fake zero-filled row was previously inserted here but produced
    # object_id=0 (the background label), creating a spurious organoid row that
    # propagated through all downstream stages.
    pass

In [13]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (55, 2668)


In [14]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [15]:
nucleocentric_df

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature90,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99
0,257,C4-1,-0.076046,-0.195801,0.099514,-0.077355,-0.087526,0.241190,0.049520,-0.119339,...,0.200481,4.058367,-5.435637,6.323337,6.208817,2.223289,0.309957,4.362774,1.527360,4.466059
1,514,C4-1,-0.393783,-0.264406,0.186608,-0.027248,-0.126305,0.140763,0.055341,-0.074079,...,1.480416,2.492021,-7.538130,3.304373,4.828958,3.199276,3.434937,3.212957,2.097705,0.973808
2,771,C4-1,-0.087094,-0.134559,0.149383,0.022436,-0.080138,0.239864,0.091013,-0.101817,...,0.671647,3.752697,-9.147327,-0.890167,5.157977,0.981803,1.429000,4.665015,0.730875,-1.203775
3,1028,C4-1,-0.203926,-0.255455,0.170370,-0.029354,-0.167938,0.139980,0.130247,-0.072525,...,0.857458,1.644536,-11.373195,-2.486522,7.482898,0.459792,-0.277286,5.905641,-2.663505,-0.668626
4,1285,C4-1,-0.201375,-0.211282,0.138028,-0.002726,-0.114167,0.264720,0.034140,-0.104014,...,1.908465,5.684974,-6.584369,5.397185,2.600838,1.398547,-0.746936,4.803525,3.831946,2.683017
5,1542,C4-1,-0.229243,-0.233788,0.195153,0.011536,-0.054042,0.222492,0.103208,-0.136733,...,2.326077,-1.194113,-8.979560,0.146452,5.144123,0.502125,2.878343,0.258343,-1.997843,0.652250
6,1799,C4-1,-0.023157,-0.118938,0.183556,0.013569,0.049664,0.242972,0.089443,-0.188933,...,-0.253793,5.795920,-6.820550,8.272780,6.055843,5.177536,-0.291035,4.130966,-0.290380,4.457836
7,2056,C4-1,-0.348012,-0.238219,0.172912,0.025692,-0.143151,0.176840,0.029272,-0.136716,...,-0.490033,3.404026,-8.528710,2.843495,8.734609,1.552848,3.269693,1.824402,1.152207,1.022455
8,2313,C4-1,-0.165853,-0.246184,0.127846,-0.066622,-0.091033,0.219009,0.115145,-0.164162,...,0.785775,-0.645982,-9.982414,-1.571068,8.601132,2.439460,2.961164,3.292048,-2.053726,1.355838
9,2570,C4-1,-0.256639,-0.205594,0.185018,-0.061629,-0.057990,0.209580,0.066522,-0.163415,...,1.542396,-0.397899,-11.346574,-1.387432,9.306607,0.691234,-0.000417,3.014294,-2.151962,0.507908


In [16]:
# Propagate ParentOrganoid to nucleocentric profiles.
# Nucleocentric objects share object_id with their parent nucleus, so joining
# on object_id + image_set carries the organoid assignment through.
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

## Get single cell and organoid relationships and spatial distributions

In [17]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    # "area" (CellProfiler) or "volumesizeshape" (ZEDProfiler) -- see the
    # x_y_z_sc_colnames cell above for why both are accepted.
    if ("area" in x.lower() or "volumesizeshape" in x.lower()) and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if ("area" in x.lower() or "volumesizeshape" in x.lower())
    and ("min" in x.lower() or "max" in x.lower())
]

In [18]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    # x_y_z_sc_colnames is alphabetically sorted: [CenterX, CenterY, CenterZ]
    # so index [0]=X, [1]=Y, [2]=Z. The dict remaps them to named z/y/x keys.
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

In [19]:
# Concatenate per-organoid shell results and rename columns to standard feature name format.
# Added columns (under Nuclei_NoChannel_Neighbors_*):
#   ShellAssignments            — shell index (1=innermost, N=outermost) for each cell
#   DistancesFromCenter         — Mahalanobis distance from organoid centroid
#   DistancesFromExterior       — distance from the outermost shell boundary
#   NormalizedDistancesFromCenter — DistancesFromCenter normalized to [0, 1]
#   ShellsUsed                  — total number of shells actually assigned (may be < 4
#                                 if too few cells to fill all shells)
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [20]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [21]:
organoid_profile_df.to_parquet(organoid_profile_handcrafted_output_path, index=False)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C4-1,55,27421038.0,711.460877,936.634612,19.864556,50391450.0,267,1082,...,35.276193,36.687325,35.246483,36.669919,35.24153,35.241639,35.290421,36.683411,35.280867,35.286644


In [22]:
sc_profile_with_shells_df.to_parquet(sc_profile_handcrafted_output_path, index=False)
sc_profile_with_shells_df.head()

,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_DNA_Texture_Variance-3-09-256,Cytoplasm_DNA_Texture_Variance-3-10-256,Cytoplasm_DNA_Texture_Variance-3-11-256,Cytoplasm_DNA_Texture_Variance-3-12-256,ParentOrganoid,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,257,C4-1,7480.0,881.128610,443.802139,1.488770,10200.0,857,907,418,...,0.0,0.0,0.0,0.0,1,3,2.115407,0.543768,0.795512,4
1,514,C4-1,37965.0,570.619781,889.462874,4.402107,53088.0,533,612,848,...,0.0,0.0,0.0,0.0,1,2,1.624583,1.034592,0.610935,4
2,771,C4-1,45736.0,630.840716,980.077226,4.681083,80442.0,576,685,939,...,0.0,0.0,0.0,0.0,1,2,1.398383,1.260792,0.525871,4
3,1028,C4-1,32804.0,513.994940,1233.819229,3.297586,55692.0,471,562,1178,...,0.0,0.0,0.0,0.0,1,3,2.027695,0.631480,0.762528,4
4,1285,C4-1,78836.0,805.882478,657.815478,9.888135,139956.0,752,859,604,...,0.0,0.0,0.0,0.0,1,1,1.170524,1.488651,0.440183,4


In [23]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,257,C4-1,-0.076046,-0.195801,0.099514,-0.077355,-0.087526,0.241190,0.049520,-0.119339,...,4.058367,-5.435637,6.323337,6.208817,2.223289,0.309957,4.362774,1.527360,4.466059,1
1,514,C4-1,-0.393783,-0.264406,0.186608,-0.027248,-0.126305,0.140763,0.055341,-0.074079,...,2.492021,-7.538130,3.304373,4.828958,3.199276,3.434937,3.212957,2.097705,0.973808,1
2,771,C4-1,-0.087094,-0.134559,0.149383,0.022436,-0.080138,0.239864,0.091013,-0.101817,...,3.752697,-9.147327,-0.890167,5.157977,0.981803,1.429000,4.665015,0.730875,-1.203775,1
3,1028,C4-1,-0.203926,-0.255455,0.170370,-0.029354,-0.167938,0.139980,0.130247,-0.072525,...,1.644536,-11.373195,-2.486522,7.482898,0.459792,-0.277286,5.905641,-2.663505,-0.668626,1
4,1285,C4-1,-0.201375,-0.211282,0.138028,-0.002726,-0.114167,0.264720,0.034140,-0.104014,...,5.684974,-6.584369,5.397185,2.600838,1.398547,-0.746936,4.803525,3.831946,2.683017,1


In [24]:
sc_profile_sammed_df.to_parquet(sc_profile_sammed_output_path, index=False)
sc_profile_sammed_df.head()

,Nuclei_ER_SAMMed3D_Feature-cls0,Nuclei_ER_SAMMed3D_Feature-cls1,Nuclei_ER_SAMMed3D_Feature-cls10,Nuclei_ER_SAMMed3D_Feature-cls100,Nuclei_ER_SAMMed3D_Feature-cls101,Nuclei_ER_SAMMed3D_Feature-cls102,Nuclei_ER_SAMMed3D_Feature-cls103,Nuclei_ER_SAMMed3D_Feature-cls104,Nuclei_ER_SAMMed3D_Feature-cls105,Nuclei_ER_SAMMed3D_Feature-cls106,...,Cytoplasm_Mito_SAMMed3D_Feature-global93,Cytoplasm_Mito_SAMMed3D_Feature-global94,Cytoplasm_Mito_SAMMed3D_Feature-global95,Cytoplasm_Mito_SAMMed3D_Feature-global96,Cytoplasm_Mito_SAMMed3D_Feature-global97,Cytoplasm_Mito_SAMMed3D_Feature-global98,Cytoplasm_Mito_SAMMed3D_Feature-global99,object_id,image_set,ParentOrganoid
0,-0.218772,-0.299993,0.240242,0.002052,-0.162141,0.178112,0.049978,-0.099201,-0.214322,-0.041488,...,-0.011200,0.054424,-0.031889,0.169822,0.008458,0.309632,0.000197,257,C4-1,1
1,-0.198356,-0.303171,0.240664,0.002708,-0.159029,0.180547,0.066317,-0.105523,-0.202614,-0.029127,...,-0.011056,0.050694,-0.022078,0.161544,0.032912,0.266970,0.046388,514,C4-1,1
2,-0.219426,-0.298148,0.244776,0.003304,-0.165985,0.174994,0.047414,-0.095766,-0.212679,-0.038359,...,-0.011023,0.053869,-0.029180,0.166843,0.022319,0.275425,0.042980,771,C4-1,1
3,-0.221368,-0.306260,0.241209,0.004705,-0.167875,0.179371,0.052670,-0.100798,-0.209155,-0.041253,...,-0.011145,0.055753,-0.037058,0.170288,0.018553,0.317224,0.007873,1028,C4-1,1
4,-0.223810,-0.324177,0.237235,-0.021822,-0.142075,0.171788,0.073343,-0.101108,-0.201399,-0.033006,...,-0.011100,0.048632,-0.013628,0.158634,0.021062,0.252813,0.049982,1285,C4-1,1


In [25]:
organoid_profile_sammed_df.to_parquet(organoid_profile_sammed_output_path, index=False)
organoid_profile_sammed_df.head()

,Organoid_DNA_SAMMed3D_Feature-cls0,Organoid_DNA_SAMMed3D_Feature-cls1,Organoid_DNA_SAMMed3D_Feature-cls10,Organoid_DNA_SAMMed3D_Feature-cls100,Organoid_DNA_SAMMed3D_Feature-cls101,Organoid_DNA_SAMMed3D_Feature-cls102,Organoid_DNA_SAMMed3D_Feature-cls103,Organoid_DNA_SAMMed3D_Feature-cls104,Organoid_DNA_SAMMed3D_Feature-cls105,Organoid_DNA_SAMMed3D_Feature-cls106,...,Organoid_ER_SAMMed3D_Feature-global93,Organoid_ER_SAMMed3D_Feature-global94,Organoid_ER_SAMMed3D_Feature-global95,Organoid_ER_SAMMed3D_Feature-global96,Organoid_ER_SAMMed3D_Feature-global97,Organoid_ER_SAMMed3D_Feature-global98,Organoid_ER_SAMMed3D_Feature-global99,object_id,image_set,OrganoidSingleCellCount
0,-0.358824,-0.311389,0.243889,0.023333,-0.194006,0.120318,0.01531,-0.045324,-0.207608,0.019166,...,-0.010522,0.056427,-0.012158,0.063217,0.037827,0.173121,0.068899,1,C4-1,55
